# Actividad 5 AI SupportAssist
## Paso 2 Configuración y validación de MLflow

El paso 1 creó el repositorio y su estructura. En este paso se configura **seguimiento local con SQLite y archivos de artefactos**. Se registra una ejecución de prueba, se cierra y después se recuperan sus parámetros, métricas y archivos para verificar la persistencia.

**Alcance:** solo configuración. `valor_prueba` y `artefactos_esperados` son valores de control del API; no representan exactitud, F1 ni desempeño de un modelo. No se selecciona un dataset ni se entrena.

Ejecuta las celdas en orden en Colab con CPU. La primera instalación requiere internet. No necesitas tokens, servicios de pago ni un servidor remoto.


### 1 Obtener el repositorio
Se descarga el código de `efrainkun-ai/Act5_GP`. Si la carpeta ya existe se reutiliza. Antes de retomar una copia antigua, sincronízala con GitHub conservando tus cambios. La ejecución registrará el commit utilizado y el hash del módulo de configuración.


In [1]:
import sys, subprocess, json
from pathlib import Path
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/efrainkun-ai/Act5_GP.git'
REPO_DIR = Path('/content/Act5_GP') if IN_COLAB else Path.cwd()
if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
assert (REPO_DIR/'fuentes'/'mlflow_config.py').is_file(), 'Abre el notebook desde una copia actualizada de Act5_GP.'
print('Repositorio:', REPO_DIR)
print(subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())


Repositorio: /workspace/scratch/bda85845160f/Act5_GP
d9f0e2a


### 2 Instalar dependencias
Se fija MLflow 3.5.0 junto con versiones compatibles de SQLAlchemy y packaging. La prueba se ejecuta en un proceso Python nuevo para utilizar las dependencias recién instaladas. En ejecución local instala primero `python -m pip install -r requirements.txt`.


In [2]:
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR/'requirements.txt')], check=True)
else:
    print('Se utiliza el entorno local previamente instalado.')
print('Dependencias listas para la verificación.')


Se utiliza el entorno local previamente instalado.
Dependencias listas para la verificación.


### 3 Elegir la ubicación del seguimiento local
En Colab, `GUARDAR_EN_DRIVE=True` conserva la base y los artefactos en Google Drive. Esto sigue siendo seguimiento local: no se configura un servidor remoto. Si eliges `False`, el almacenamiento temporal del entorno puede perderse al cerrar la sesión. La base y el directorio de artefactos deben permanecer juntos en su ubicación de trabajo.


In [3]:
GUARDAR_EN_DRIVE = True
if IN_COLAB and GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    TRACKING_DIR = Path('/content/drive/MyDrive/Act5_GP/mlflow_local')
else:
    TRACKING_DIR = REPO_DIR/'.mlflow_local'
TRACKING_DIR.mkdir(parents=True, exist_ok=True)
EVIDENCIA = TRACKING_DIR/'evidencias'/'mlflow_verificacion.json'
print('Seguimiento local:', TRACKING_DIR)
print('Evidencia:', EVIDENCIA)


Seguimiento local: /workspace/scratch/bda85845160f/Act5_GP/.mlflow_local
Evidencia: /workspace/scratch/bda85845160f/Act5_GP/.mlflow_local/evidencias/mlflow_verificacion.json


### 4 Registrar y validar
El script configura el experimento `Act5_GP_Configuracion`. Cada ejecución registra parámetros de proyecto, dos métricas de control y dos artefactos (`verificacion.txt` y `entorno.json`). Tras cerrar el run, un cliente vuelve a leer los datos, descarga los artefactos y compara su contenido byte a byte. Si alguna comprobación falla se genera un error y no se declara éxito.


In [4]:
proceso = subprocess.run(
    [sys.executable, str(REPO_DIR/'fuentes'/'train.py'),
     '--verificar-mlflow', '--tracking-dir', str(TRACKING_DIR), '--evidencia', str(EVIDENCIA)],
    capture_output=True, text=True,
)
print(proceso.stdout)
if proceso.returncode != 0:
    print(proceso.stderr)
    raise RuntimeError('MLflow no superó la verificación. Revisa el error anterior.')


{
  "resultado": "VALIDACION_CORRECTA",
  "fecha_utc": "2026-10-16T02:49:19.359084+00:00",
  "experiment_name": "Act5_GP_Configuracion",
  "experiment_id": "1",
  "run_id": "c393d44aa13841dc9f1fcc2c2f35f547",
  "estado": "FINISHED",
  "parametros": {
    "proyecto": "AI-SupportAssist",
    "repositorio": "Act5_GP",
    "etapa": "2_configuracion_mlflow",
    "semilla": "42",
    "tipo_prueba": "infraestructura_sin_entrenamiento"
  },
  "metricas": {
    "valor_prueba": 1.0,
    "artefactos_esperados": 2.0
  },
  "artefactos_sha256": {
    "verificacion.txt": "69f60dc999a6a317493c6af88e46e184c7f9a608adac553530dc0af758f3cfb6",
    "entorno.json": "41d2e9dce458ab943b91711e0d9824ea47deeb1c2bc2bb6b3875597c2777c9d8"
  },
  "controles": {
    "estado_finished": true,
    "parametros_recuperados": true,
    "metricas_recuperadas": true,
    "artefactos_listados": true,
    "contenido_verificacion.txt": true,
    "contenido_entorno.json": true
  },
  "entorno": {
    "python": "3.12.14",
    "ml

### 5 Revisar la evidencia
La evidencia solo se muestra después de una ejecución satisfactoria. Comprueba `VALIDACION_CORRECTA`, estado `FINISHED`, parámetros, métricas y controles en `true`. Cada repetición genera un run nuevo en el mismo experimento.


In [5]:
resultado = json.loads(EVIDENCIA.read_text(encoding='utf-8'))
assert resultado['resultado'] == 'VALIDACION_CORRECTA'
assert resultado['estado'] == 'FINISHED'
assert all(resultado['controles'].values())
print('Experimento:', resultado['experiment_name'])
print('Run ID:', resultado['run_id'])
print('Estado:', resultado['estado'])
print('Parámetros:', resultado['parametros'])
print('Métricas de control:', resultado['metricas'])
print('Archivos:', list(resultado['artefactos_sha256']))
print('Validaciones:', resultado['controles'])
print('CONFIGURACIÓN VALIDADA. El entrenamiento corresponde a una etapa posterior.')


Experimento: Act5_GP_Configuracion
Run ID: c393d44aa13841dc9f1fcc2c2f35f547
Estado: FINISHED
Parámetros: {'proyecto': 'AI-SupportAssist', 'repositorio': 'Act5_GP', 'etapa': '2_configuracion_mlflow', 'semilla': '42', 'tipo_prueba': 'infraestructura_sin_entrenamiento'}
Métricas de control: {'valor_prueba': 1.0, 'artefactos_esperados': 2.0}
Archivos: ['verificacion.txt', 'entorno.json']
Validaciones: {'estado_finished': True, 'parametros_recuperados': True, 'metricas_recuperadas': True, 'artefactos_listados': True, 'contenido_verificacion.txt': True, 'contenido_entorno.json': True}
CONFIGURACIÓN VALIDADA. El entrenamiento corresponde a una etapa posterior.


### Consulta visual opcional
En un equipo local, desde la raíz del proyecto, ejecuta:

```bash
mlflow ui --backend-store-uri sqlite:///.mlflow_local/mlflow.db --host 127.0.0.1 --port 5000
```

Abre `http://127.0.0.1:5000`, selecciona `Act5_GP_Configuracion` y revisa la ejecución. En Colab la consulta y descarga mediante el cliente ya se verifican sin necesidad de publicar un túnel. El JSON generado en tu Drive es la evidencia de tu propia ejecución; el archivo de evidencia del repositorio procede de una validación local anterior.

Referencias: [MLflow Tracking Quickstart](https://mlflow.org/docs/latest/ml/tracking/quickstart/) y [API de seguimiento](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.html).
